In [ ]:
# %% Deep learning - Section 20.193
#    CNN project 4: psychometric functions in CNNs
#    1) Generate a dataset of 1x30x30 images with gaussian noise and some random
#       vertical or horizontal lines
#    2) Train a CNN only on these perfectly horizontal or vertical lines and
#       test on a similar dataset; performace should easily reach 100% even in
#       just 10 epochs
#    3) Generate another test dataset with lines on a clear background, the
#       lines should have all the different slopes allowed by a 30x30
#       resolution, have a look at skimage.draw.line_aa
#    4) Test the model trained only on horizontal and vertical lines
#    5) Plot the actual slope against the probability of being classified as
#       vertical; plot some examples by color-coding the model classification

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [7]:
# %% Generate data

# 2D Gaussian params
n_images = 2000
img_size = 30

# Preallocate tensors for images (N,channels,size,size)
images = torch.zeros(n_images,1,img_size,img_size)
labels = torch.zeros(n_images,1)

# Generate images
for i in range(n_images):

    # Layer some noise
    G  = np.random.randn(img_size,img_size) / 1

    # Add random bar randomly
    i1 = np.random.choice(np.arange(2,28))
    i2 = np.random.choice(np.arange(2,6))

    # Horizontal
    if np.random.randn()>0:
        G[i1:i1+i2,] = 1
    # Vertical
    else:
        G[:,i1:i1+i2] = 1
        labels[i] = 1

    # Add to tensor
    images[i,:,:,:] = torch.Tensor(G).view(1,img_size,img_size)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(2,7,figsize=(phi*7,7))

for i,ax in enumerate(axs.flatten()):

    pic = np.random.randint(n_images)
    G   = np.squeeze(images[pic,:,:])

    ax.imshow(G,vmin=-2,vmax=2,cmap='jet',extent=[-4,4,-4,4],origin='upper',interpolation='bilinear')

    title = 'Horizontal' if labels[pic].item()==0 else 'Vertical'
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()

plt.savefig('figure15_cnn_project_4.png')
plt.show()
files.download('figure15_cnn_project_4.png')


In [14]:
# %% Create train and test datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size   = 16
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [ ]:
# %% Check sizes

# Should be (N,channels,width,height) and (N,3)
print( train_loader.dataset.tensors[0].shape )
print( train_loader.dataset.tensors[1].shape )


In [26]:
# %% Function to generate the model

def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Layers with nn.Sequential (activation function is treated as layer)
            self.model = nn.Sequential( nn.Conv2d(1,6,3,padding=1),     # out size: (30+2*1-3)/1 + 1 = 30
                                        nn.ReLU(),
                                        nn.MaxPool2d(2,2),              # out size: 30/2 = 15
                                        nn.Conv2d(6,4,3,padding=1),     # out size: (15+2*1-3)/1 + 1 = 15
                                        nn.ReLU(),
                                        nn.MaxPool2d(2,2),              # out size: 15/2 = 7
                                        nn.Flatten(),                   # vectorise
                                        nn.Linear(7*7*4,50),
                                        nn.ReLU(),
                                        nn.Linear(50,1)                 # out size: 2
                                        )

        def forward(self,x):
            return self.model(x)

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function (MSELoss for continous predictions)
    loss_fun = nn.BCEWithLogitsLoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

CNN,loss_fun,optimizer = gen_model()

X,y  = next(iter(train_loader))
yHat = CNN(X)

# Check sizes of output and target variable
print()
print(yHat.shape), print()
print(y.shape), print()

# Check loss
loss = loss_fun(yHat,y)
print(loss)


In [ ]:
# %% Check all the parameters in the model

summary(CNN,(1,img_size,img_size))


In [38]:
# %% Function to train the model

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 10
    CNN,loss_fun,optimizer = gen_model()

    train_losses = []
    test_losses  = []
    train_acc    = []
    test_acc     = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_loss = []
        batch_acc  = []

        for X,y in train_loader:

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and accuracy from this batch
            batch_loss.append(loss.item())

            probs = torch.sigmoid(yHat)
            preds = (probs > 0.5).float()
            batch_acc.append(torch.mean((preds == y).float()).item())

        train_losses.append( np.mean(batch_loss) )
        train_acc.append( 100*np.mean(batch_acc) )

        # Test performance
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            test_losses.append(loss.item())

            probs = torch.sigmoid(yHat)
            preds = (probs > 0.5).float()
            test_acc.append(100 * torch.mean((preds == y).float()).item())

        CNN.train()

    return train_acc,test_acc,train_losses,test_losses,CNN


In [39]:
# %% Run the model

# Takes a dew secs
train_acc,test_acc,train_losses,test_losses,CNN = train_model()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(1,2,figsize=(1.5*phi*6,6))

axs[0].plot(train_losses,'s-',label='Train')
axs[0].plot(test_losses,'o-',label='Test')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss (BCEWithLogitsLoss)')
axs[0].legend()
axs[0].set_title(f'Model loss (final test loss: {test_losses[-1]:.2f})')

axs[1].plot(train_acc,'s-',label='Train')
axs[1].plot(test_acc,'o-',label='Test')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Accuracy (%)')
axs[1].legend()
axs[1].set_title(f'Model accuracy (final test accuracy: {test_acc[-1]:.2f}%)')

plt.savefig('figure16_cnn_project_4.png')
plt.show()
files.download('figure16_cnn_project_4.png')


In [44]:
# %% Generate test dataset with arbitrary orientations

from skimage.draw import line_aa

imgs_angle = np.zeros((2*img_size,1,img_size,img_size))
slopes     = np.zeros(2*img_size)

for i in range(img_size):

    # |slope| < 1
    p1 = (0, i)
    p2 = (img_size-1, img_size-1-i)

    dy = p2[0] - p1[0]
    dx = p2[1] - p1[1]
    slopes[i] = dy/dx if dx != 0 else np.inf

    rr, cc, val = line_aa(p1[0], p1[1], p2[0], p2[1])
    imgs_angle[i,0,rr,cc] = val

    # |slope| > 1
    p1 = (i, 0)
    p2 = (img_size-1-i, img_size-1)

    dy = p2[0] - p1[0]
    dx = p2[1] - p1[1]
    slopes[i+img_size] = dy/dx if dx != 0 else np.inf

    rr, cc, val = line_aa(p1[0], p1[1], p2[0], p2[1])
    imgs_angle[i+img_size,0,rr,cc] = val


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig, ax = plt.subplots(6,10,figsize=(phi*7,7))
ax = ax.flatten()

for i in range(60):
    ax[i].imshow(imgs_angle[i,0,:,:], cmap='plasma', origin='lower')
    ax[i].set_title(f"slope = {slopes[i]:.2f}", fontsize=8)
    ax[i].axis('off')

plt.tight_layout()

plt.savefig('figure17_cnn_project_4.png')
plt.show()
files.download('figure17_cnn_project_4.png')


In [48]:
# %% Pass new dataset through CNN

# To tensor
X_new = torch.tensor(imgs_angle, dtype=torch.float32)

CNN.eval()

with torch.no_grad():
    logits = CNN(X_new)
    probs  = torch.sigmoid(logits)
    preds  = (probs > 0.5).float()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(figsize=(8,5))

for i in range(len(slopes)):

    p = probs[i].item()
    slope = slopes[i]
    angle = np.arctan(slope)

    is_vertical = p > 0.5

    if is_vertical:
        color = 'orange'
        label_text = 'vertical'
    else:
        color = 'blue'
        label_text = 'horizontal'

    ax.scatter(slope, p, color=color, alpha=0.7)
    # ax.scatter(angle, p, color=color, alpha=0.7)

ax.axhline(0.5, linestyle=':',color='grey')

# xticks = [-np.pi/2, -np.pi/4, 0, np.pi/4, np.pi/2]
# xtick_labels = [
#     r"$-\frac{\pi}{2}$",
#     r"$-\frac{\pi}{4}$",
#     r"$0$",
#     r"$\frac{\pi}{4}$",
#     r"$\frac{\pi}{2}$"
# ]

# ax.set_xticks(xticks)
# ax.set_xticklabels(xtick_labels)

ax.set_xlabel('Slope')
# ax.set_xlabel('Angle (radians)')
ax.set_ylabel('P(vertical)')

plt.tight_layout()

plt.savefig('figure18_cnn_project_4.png')
plt.show()
files.download('figure18_cnn_project_4.png')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(6,10,figsize=(phi*7,7))
ax = ax.flatten()

for i in range(60):

    # Get prediction
    p = probs[i].item()
    is_vertical = p > 0.5

    # Choose colormap and label
    if is_vertical:
        cmap = 'Oranges'
        label = 'pred. vertical'
    else:
        cmap = 'Blues'
        label = 'pred. horizontal'

    ax[i].imshow(imgs_angle[i,0,:,:], cmap=cmap, origin='lower')
    ax[i].set_title(label, fontsize=8)
    ax[i].axis('off')

plt.tight_layout()

plt.savefig('figure20_cnn_project_4.png')
plt.show()
files.download('figure20_cnn_project_4.png')
